# Lab 24-05 · Bandwidth, Credits and Fabrics

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import links

## Part 1 · Bandwidth

*Predict* the payload bandwidth of PCIe 4.0 x8 with 128-byte TLPs.

In [2]:
for g in links.PCIE:
    print(f"PCIe {g}: {links.pcie_bw(g):6.3f} GB/s per lane per direction; x16 {links.pcie_bw(g, 16):7.2f} GB/s")
bw = links.pcie_bw("4.0", 8)
print(f"PCIe 4.0 x8: {bw:.2f} GB/s after encoding; {bw * links.tlp_efficiency(128):.2f} GB/s payload with 128-byte TLPs")
for p in (64, 128, 256, 512):
    print(f"  TLP payload {p:3d} B: efficiency {100*links.tlp_efficiency(p):.1f}%")

PCIe 1.0:  0.250 GB/s per lane per direction; x16    4.00 GB/s
PCIe 2.0:  0.500 GB/s per lane per direction; x16    8.00 GB/s
PCIe 3.0:  0.985 GB/s per lane per direction; x16   15.75 GB/s
PCIe 4.0:  1.969 GB/s per lane per direction; x16   31.51 GB/s
PCIe 5.0:  3.938 GB/s per lane per direction; x16   63.02 GB/s
PCIe 6.0:  7.375 GB/s per lane per direction; x16  118.00 GB/s
PCIe 7.0: 14.750 GB/s per lane per direction; x16  236.00 GB/s
PCIe 4.0 x8: 15.75 GB/s after encoding; 13.27 GB/s payload with 128-byte TLPs
  TLP payload  64 B: efficiency 72.7%
  TLP payload 128 B: efficiency 84.2%
  TLP payload 256 B: efficiency 91.4%
  TLP payload 512 B: efficiency 95.5%


## Part 2 · Credits

*Predict* the throughput of a 128 GB/s link with a 32 KB buffer and a 500 ns round trip.

In [3]:
bufs = np.arange(8_000, 160_001, 8_000)
plt.figure()
for rtt in (250, 500, 1000):
    thr = [links.simulate_credits(128, int(b), rtt) for b in bufs]
    plt.plot(bufs / 1e3, thr, "o-", ms=3, label=f"RTT {rtt} ns (BDP {links.bdp_bytes(128, rtt)/1e3:.0f} KB)")
print(f"32 KB at 500 ns: {links.simulate_credits(128, 32_000, 500):.1f} GB/s "
      f"(rule: {min(128, 32_000/500):.1f} GB/s)")
plt.xlabel("receive buffer (KB)"); plt.ylabel("throughput (GB/s)"); plt.legend(); plt.grid(True)
plt.show()

32 KB at 500 ns: 64.0 GB/s (rule: 64.0 GB/s)


## Part 3 · Fabrics

*Predict* the two-tier host count with 102.4 Tb/s switches at 800 Gb/s.

In [4]:
k = links.radix(102.4, 800)
print(f"radix {k}: two tiers {links.two_tier_hosts(k)} hosts, three tiers {links.three_tier_hosts(k)} hosts")
print(f"200 Gb/s Ethernet lane: {links.ethernet_lane_rate(200):.2f} Gb/s")
print(f"replay buffer, 800 Gb/s and 1 us: {links.replay_buffer_bytes(800, 1)/1e3:.0f} KB")

radix 128: two tiers 8192 hosts, three tiers 524288 hosts
200 Gb/s Ethernet lane: 212.50 Gb/s
replay buffer, 800 Gb/s and 1 us: 100 KB
